In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir(os.path.dirname(os.getcwd()))
    sys.path.append(os.getcwd())

full_runs_dir = "../logs_cluster/logs/full_runs/"



In [ ]:
import re
from pathlib import Path

# All requested runs under march_old
run_dirs = [
    "2026_02_26_long_warmup_kodak_one_by_one_0",
    "2026_02_26_long_warmup_kodak_one_by_one_1",
    "2026_02_26_long_warmup_kodak_one_by_one_2",
    "2026_03_01_long_warmup_kodak_two_by_two_0",
    "2026_03_01_long_warmup_kodak_two_by_two_1",
    "2026_03_01_long_warmup_kodak_two_by_two_2",
    "2026_03_01_long_warmup_kodak_three_by_three_0",
    "2026_03_01_long_warmup_kodak_three_by_three_1",
    "2026_03_01_long_warmup_kodak_three_by_three_2",
]

kodim_re = re.compile(r"kodim(\d{2})", re.IGNORECASE)
num_pat = r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
final_re = re.compile(
    r"Final results after quantization:\s*"
    rf"Loss:\s*{num_pat},\s*"
    rf"Rate NN:\s*{num_pat},\s*"
    rf"Rate Latent:\s*{num_pat},\s*"
    rf"Rate Img:\s*{num_pat}"
)


def parse_log(log_path: Path):
    text = log_path.read_text(encoding="utf-8", errors="replace")

    # First important section (from your example) — we require these markers.
    has_processing = "Processing image" in text
    has_mac = "Total MAC per pixel:" in text

    # Final metrics line.
    final_match = final_re.search(text)

    missing = []
    if not has_processing:
        missing.append("'Processing image' section")
    if not has_mac:
        missing.append("'Total MAC per pixel' line")
    if final_match is None:
        missing.append("'Final results after quantization' line")

    if missing:
        print(f"[SKIP] {log_path}: missing {', '.join(missing)}")
        return None

    kodim_match = kodim_re.search(log_path.name)
    if kodim_match is None:
        print(f"[SKIP] {log_path}: missing 'kodimXX' in filename")
        return None

    kodim_idx = int(kodim_match.group(1))
    assert final_match is not None
    loss, rate_nn, rate_latent, rate_img = map(float, final_match.groups())
    return {
        "kodim": kodim_idx,
        "loss": loss,
        "rate_nn": rate_nn,
        "rate_latent": rate_latent,
        "rate_img": rate_img,
        "source": str(log_path),
    }


march_old_root = Path(full_runs_dir) / "march_old"

for run_dir in run_dirs:
    results_dir = march_old_root / run_dir / "results"
    print(f"\n=== {run_dir} ===")

    if not results_dir.exists():
        print(f"[WARN] Missing directory: {results_dir}")
        continue

    best_by_kodim = {}
    for log_path in sorted(results_dir.glob("*.log")):
        parsed = parse_log(log_path)
        if parsed is None:
            continue

        k = parsed["kodim"]
        # If duplicates exist for the same kodimXX, keep lower final loss.
        if k not in best_by_kodim or parsed["loss"] < best_by_kodim[k]["loss"]:
            best_by_kodim[k] = parsed

    if not best_by_kodim:
        print("No valid logs found.")
        continue

    print("kodim loss rate_nn rate_latent rate_img")
    rows = [best_by_kodim[kodim] for kodim in sorted(best_by_kodim)]

    for row in rows:
        print(
            f"{row['kodim']:02d} "
            f"{row['loss']:.12g} "
            f"{row['rate_nn']:.12g} "
            f"{row['rate_latent']:.12g} "
            f"{row['rate_img']:.12g}"
        )

    n = len(rows)
    avg_loss = sum(r["loss"] for r in rows) / n
    avg_rate_nn = sum(r["rate_nn"] for r in rows) / n
    avg_rate_latent = sum(r["rate_latent"] for r in rows) / n
    avg_rate_img = sum(r["rate_img"] for r in rows) / n

    print(
        f"Average "
        f"{avg_loss:.12g} "
        f"{avg_rate_nn:.12g} "
        f"{avg_rate_latent:.12g} "
        f"{avg_rate_img:.12g}"
    )



=== 2026_02_26_long_warmup_kodak_one_by_one_0 ===
kodim loss rate_nn rate_latent rate_img
01 3.20845317841 0.0146179199219 0.213628411293 2.9802069664
02 2.87453341484 0.0167439778646 0.304467052221 2.55332255363
03 2.68411970139 0.0141728719076 0.179894655943 2.49005222321
04 2.90168714523 0.0155715942383 0.223146349192 2.6629691124
05 3.4243710041 0.0143932766385 0.168939098716 3.24103856087
06 3.06361222267 0.0149544609918 0.141575083137 2.90708255768
07 2.57130336761 0.0139295789931 0.215616345406 2.34175753593
08 3.67664837837 0.0132590399848 0.123428747058 3.53996062279
09 2.84164905548 0.0143585205078 0.0748956725001 2.75239491463
10 2.81587171555 0.013680352105 0.0668426305056 2.73534870148
11 2.87904763222 0.0143203735352 0.145077541471 2.71964979172
12 2.73441576958 0.0143212212457 0.0773971304297 2.64269733429
13 3.67074608803 0.0142695109049 0.28593441844 3.37054204941
14 3.22553944588 0.0145034790039 0.116165801883 3.09487009048
15 2.71307969093 0.0163633558485 0.11124110